<a href="https://colab.research.google.com/github/Jun-1112/FYP-project-Trunk-and-weed-detection-for-agricultural-usage/blob/main/Object_detection(trunk).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#setup
!pip install ultralytics -q

from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
# To import the CVAT-refined trunk dataset from Drive
refined_zip = '/content/drive/MyDrive/Colab Notebooks/predicted_annotations_full.zip'
extract_dir = '/content/cvat_extracted'

!unzip -q "{refined_zip}" -d "{extract_dir}"
print("Extracted refined dataset:", os.listdir(extract_dir))

In [ ]:
# To build 85/15 train/val split and write data.yaml
import glob, random, shutil, yaml

src_img_dir = '/content/cvat_extracted/images/Train'
src_lbl_dir = '/content/cvat_extracted/labels/Train'

out_root = '/content/full_dataset'
for split in ['train', 'val']:
    os.makedirs(f'{out_root}/images/{split}', exist_ok=True)
    os.makedirs(f'{out_root}/labels/{split}', exist_ok=True)

img_files = sorted(glob.glob(f'{src_img_dir}/*.jpg'))
print("Total images found:", len(img_files))

random.seed(0)
random.shuffle(img_files)
val_count = max(1, int(len(img_files) * 0.15))
val_set = set(img_files[:val_count])

for img_path in img_files:
    stem = os.path.splitext(os.path.basename(img_path))[0]
    lbl_path = f'{src_lbl_dir}/{stem}.txt'
    split = 'val' if img_path in val_set else 'train'
    shutil.copy(img_path, f'{out_root}/images/{split}/{stem}.jpg')
    if os.path.exists(lbl_path):
        shutil.copy(lbl_path, f'{out_root}/labels/{split}/{stem}.txt')
    else:
        # Frames with no trunk instance get an empty label file (valid in YOLO)
        open(f'{out_root}/labels/{split}/{stem}.txt', 'w').close()

print(f"Train: {len(img_files) - val_count}, Val: {val_count}")

data_yaml = {
    'path': '/content/full_dataset',
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'Palm Tree Trunk'}
}
with open('/content/full_dataset/data.yaml', 'w') as f:
    yaml.dump(data_yaml, f)

print(open('/content/full_dataset/data.yaml').read())

In [ ]:
# To perform checking on class distribution
from collections import Counter

counts = Counter()
for split in ['train', 'val']:
    for fname in os.listdir(f'{out_root}/labels/{split}'):
        with open(f'{out_root}/labels/{split}/{fname}') as fh:
            for line in fh:
                if line.strip():
                    counts[int(line.split()[0])] += 1

print("Instances per class id:", dict(counts))

In [ ]:
# To train the trunk detection model (YOLOv8n)
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='/content/full_dataset/data.yaml',
    epochs=150,
    imgsz=640,
    batch=16,
    patience=25,
    device=0,
    plots=True,
    project='/content/runs/detect',
    name='det_model_for_treetrunk',
    save_period=10,
)

print("Training complete. Best weights saved.")

In [ ]:
# To back up weights, plots, metrics, and config to Drive
import shutil, os

backup_dir = '/content/drive/MyDrive/Colab Notebooks/detfyp_backup'
os.makedirs(backup_dir, exist_ok=True)

run_dir = '/content/runs/detect/det_model_for_treetrunk'

# Weights
shutil.copy(f'{run_dir}/weights/best.pt', f'{backup_dir}/bestdet.pt')
shutil.copy(f'{run_dir}/weights/last.pt', f'{backup_dir}/lastdet.pt')

# Config + metrics
shutil.copy('/content/full_dataset/data.yaml', f'{backup_dir}/data.yaml')
shutil.copy(f'{run_dir}/results.csv', f'{backup_dir}/results.csv')
shutil.copy(f'{run_dir}/args.yaml', f'{backup_dir}/args.yaml')

# All generated plots and batch visualisations
for f in os.listdir(run_dir):
    if f.endswith('.png') or f.endswith('.jpg'):
        shutil.copy(f'{run_dir}/{f}', f'{backup_dir}/{f}')

print("Weights, data.yaml, results.csv, args.yaml, and all plots backed up to Drive.")